# P5 — Sequence Modeling with RNNs

**Goal:** Apply basic and advanced **RNN / LSTM / GRU, Encoder-Decoder** architectures to a sequence-based sentiment classification task.

---

## Dataset (Required)

### IMDB Movie Reviews (Binary Sentiment Classification)
**Access method (required):**
```python
from tensorflow.keras.datasets import imdb
(x_train, y_train), (x_test, y_test) = ...
```

You must create a validation split from training data and pad/truncate sequences to a fixed length.

---
## What you will implement

You will implement and compare the following **sequence models**:

- **Vanilla RNN**
- **LSTM**
- **GRU**
- **Stacked LSTM + Dropout**
- **Encoder–Decoder LSTM**
- **LSTM + GRU Hybrid**

---

## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [1]:
# ============================================================
# Q0) Environment Setup
# ============================================================

import os
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


TensorFlow: 2.20.0


---
## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q11)** and continues with the **Markdown responses (Q12–Q15)**.  
These correspond to the questions listed in the assignment description on **Canvas**. Follow the instructions provided in the **preceding Markdown cells** for each step.

### Tasks

This assignment focuses on **sequence modeling for text classification** using recurrent neural networks.

You will:

- Train and evaluate the following **sequence models**:
  - **Vanilla RNN**
  - **LSTM**
  - **GRU**

- Implement additional **advanced architectures**:
  - **Stacked LSTM + Dropout**
  - **Encoder–Decoder LSTM**
  - **LSTM + GRU Hybrid**

- Use the **IMDB Movie Reviews dataset** for **binary sentiment classification**.

- Perform a **comparative analysis** of the models, including:
  - training convergence behavior
  - validation and test performance
  - explanation of architectural differences across models

Ensure that all models are **computationally feasible** to train on **CPU-only environments** by using the recommended hyperparameters unless you have access to a GPU (e.g. Google Colab).

---

## Q1 — Load Dataset & Inspect

Use the **IMDB Movie Reviews dataset** from Keras and inspect its basic structure.

### Student Tasks

- Load the IMDB dataset using `tensorflow.keras.datasets.imdb` with a vocabulary size of **10,000 words** by frequency.

- Split the training data into **training** and **validation** sets.

- Inspect the dataset by:
  - Printing the **number of training and test samples**
  - Displaying the **label distribution (train)**
  - Printing one example **Sequence length (train)**

---

In [2]:
# ============================================================
# Question Q1 — Load IMDB Dataset (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Load the IMDB Movie Reviews dataset
# 2) Print the number of training and test examples
# 3) Check the label distribution in the training set
# 4) Inspect sequence length statistics
# ============================================================

import numpy as np
from tensorflow.keras.datasets import imdb

# TODO 1: Define vocabulary size
VOCAB_SIZE = 10000

# TODO 2: Load the IMDB dataset
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# TODO 3: Print dataset sizes
print("Train examples:", len(x_train))
print("Test examples: ", len(x_test))

# TODO 4: Print label distribution
print("Label distribution (train):", np.bincount(y_train))

# TODO 5: Compute sequence length statistics
train_lengths = np.array([len(s) for s in x_train])

print(
    "Sequence length (train): min/median/max =",
    train_lengths.min(),
    int(np.median(train_lengths)),
    train_lengths.max()
)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train examples: 25000
Test examples:  25000
Label distribution (train): [12500 12500]
Sequence length (train): min/median/max = 11 178 2494


---

## Q2 — Validation Split & Sequence Padding

Prepare the dataset for training by creating a **validation split** and converting all sequences to a **fixed length**.

### Student Tasks

- Create a **validation set** from the training data (`VAL_SIZE = 5000`).  
  Use a **deterministic split from the end of the training set**.

- Define a maximum sequence length **`MAX_LEN`** (e.g., 200–300 tokens) based on the sequence statistics observed in **Q1**.

- Apply **sequence padding and truncation** using `pad_sequences` so that all reviews have the same length:
  - Use **post-padding**
  - Use **post-truncation**

- Generate padded datasets for:
  - `x_train_pad`
  - `x_val_pad`
  - `x_test_pad`

- Keep it **consistent across all models**.

- Print the shapes of the padded arrays to confirm the preprocessing step completed successfully.

---

In [3]:
# ============================================================
# Question Q2 — Validation Split & Sequence Padding (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define MAX_LEN and validation size
# 2) Create a validation split from the training data
# 3) Pad/truncate sequences to a fixed length
# 4) Verify resulting dataset shapes
# ============================================================

from tensorflow.keras.preprocessing.sequence import pad_sequences

# TODO 1: Define sequence length and validation size
MAX_LEN = 200
VAL_SIZE = 5000

# TODO 2: Create validation split from the end of the training set
x_val, y_val = x_train[-VAL_SIZE:], y_train[-VAL_SIZE:]
x_train2, y_train2 = x_train[:-VAL_SIZE], y_train[:-VAL_SIZE]

# TODO 3: Apply padding and truncation
x_train_pad = pad_sequences(
    x_train2,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

x_val_pad = pad_sequences(
    x_val,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

x_test_pad = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

# Print dataset shapes
print("x_train_pad:", x_train_pad.shape)
print("x_val_pad:  ", x_val_pad.shape)
print("x_test_pad: ", x_test_pad.shape)

x_train_pad: (20000, 200)
x_val_pad:   (5000, 200)
x_test_pad:  (25000, 200)


---

## Q3 — Build `tf.data` Pipelines

Create efficient **data pipelines** for training, validation, and testing using **TensorFlow `tf.data`**.

### Student Tasks

- Convert the padded datasets into **TensorFlow datasets** using `tf.data.Dataset.from_tensor_slices`.

- Create datasets for:
  - **training**
  - **validation**
  - **testing**

- Apply the following pipeline steps:
  - **shuffle** the training dataset
  - **batch** the datasets using an appropriate batch size
  - use **prefetching** to improve training performance

- Ensure the pipelines are ready to be used directly in **model training with `model.fit()`**.


---

In [10]:
# ============================================================
# Question Q3 — tf.data Pipelines (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define batch size and AUTOTUNE
# 2) Create TensorFlow datasets from the padded arrays
# 3) Apply shuffle, batch, and prefetch operations
# 4) Prepare pipelines for training, validation, and testing
# ============================================================

# TODO 1: Define batch size and autotune
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

# TODO 2: Create training dataset
train_ds = tf.data.Dataset.from_tensor_slices((x_train_pad, y_train2))

# TODO 3: Apply shuffle, batch, and prefetch
train_ds = train_ds.shuffle(10000, seed=42).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 4: Create validation dataset
val_ds = tf.data.Dataset.from_tensor_slices((x_val_pad, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 5: Create test dataset
test_ds = tf.data.Dataset.from_tensor_slices((x_test_pad, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Pipelines ready.")

Pipelines ready.


---

## Q4 — Define Training Utilities

In this step, you will create **reusable helper functions** that simplify model training, evaluation, and experiment management.

These utilities will be used for **all models later in the assignment**.

### Student Tasks

1. **Create training callbacks**

   Define a function that returns commonly used training callbacks, including:

   - **EarlyStopping** to stop training when validation performance stops improving.
   - **ReduceLROnPlateau** to automatically reduce the learning rate when validation loss plateaus.
   - **ModelCheckpoint** to save the **best-performing model** during training.

2. **Define a model compilation function**

   Implement a function that compiles a model using:

   - **Adam optimizer**
   - **Binary cross-entropy loss** for sentiment classification
   - **Accuracy** as the evaluation metric

3. **Create a training function**

   Implement a function that trains a model using:

   - the **training dataset**
   - the **validation dataset**
   - the callbacks defined above

4. **Create an evaluation function**

   Implement a function that evaluates a trained model on a dataset and reports:

   - **loss**
   - **accuracy**

These functions will help keep the notebook **organized, reusable, and consistent across experiments**.

---

In [13]:
# ============================================================
# Question Q4 — Training Utilities (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define callbacks for training
# 2) Compile the model with optimizer, loss, and metrics
# 3) Train the model using training and validation datasets
# 4) Evaluate the trained model on a dataset
# ============================================================

# TODO 1: Define training callbacks
def build_callbacks(run_name: str):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"{run_name}.keras",
            monitor="val_accuracy",
            save_best_only=True,
        ),
    ]


# TODO 2: Compile model
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryFocalCrossentropy(from_logits=False),
        metrics=["accuracy"],
    )
    return model


# TODO 3: Train model
def train_model(model, run_name: str, epochs=8):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=build_callbacks(run_name),
        verbose=1,
    )
    return history


# TODO 4: Evaluate model
def evaluate_model(model, name: str, ds):
    loss, acc = model.evaluate(ds, verbose=0)
    print(f"{name}: loss={loss:.4f}, acc={acc:.4f}")
    return loss, acc

---

## Q5 — Model A: Vanilla RNN

Build a **Vanilla RNN** model for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add a **SimpleRNN layer** for sequence processing.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.

---

In [11]:
# ============================================================
# Question Q5 — Model A: Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define embedding dimension and RNN units
# 2) Build a Sequential RNN model
# 3) Add Input, Embedding, RNN, and Dense layers
# 4) Compile the model
# 5) Display the model summary
# ============================================================

from tensorflow.keras import layers

# TODO 1: Define model hyperparameters (e.g., 128, 128)
EMBED_DIM = 128
RNN_UNITS = 128

# TODO 2: Build the Sequential model
rnn_model = tf.keras.Sequential([

    # TODO 3: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 4: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 5: Simple RNN layer
    layers.SimpleRNN(RNN_UNITS),

    # TODO 6: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="vanilla_rnn")

# TODO 7: Compile the model
rnn_model = compile_model(rnn_model, lr=1e-3)

# Print model summary
rnn_model.summary()

Model: "vanilla_rnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# ============================================================
# Train and Evaluate Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the Vanilla RNN model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the RNN model
history_rnn = train_model(rnn_model, run_name="proj5_vanilla_rnn", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(rnn_model, "Validation (RNN)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(rnn_model, "Test (RNN)", test_ds)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 42s 133ms/step - accuracy: 0.6133 - loss: 0.1616 - val_accuracy: 0.5050 - val_loss: 0.1749 - learning_rate: 6.2500e-05
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 128ms/step - accuracy: 0.6172 - loss: 0.1587 - val_accuracy: 0.5036 - val_loss: 0.1758 - learning_rate: 6.2500e-05
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 128ms/step - accuracy: 0.6257 - loss: 0.1555 - val_accuracy: 0.5060 - val_loss: 0.1766 - learning_rate: 3.1250e-05
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 127ms/step - accuracy: 0.6299 - loss: 0.1536 - val_accuracy: 0.5054 - val_loss: 0.1771 - learning_rate: 1.5625e-05
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 127ms/step - accuracy: 0.6303 - loss: 0.1527 - val_accuracy: 0.5042 - val_loss: 0.1772 - learning_rate: 7.8125e-06
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 128ms/step - accuracy: 0.6332 - loss: 0.1523 - val_accuracy: 0.5038 - val_loss: 0.1773 - learning_rate: 3.9063e-06
Validation (RNN): loss=0.1766, acc=0.5060
Test (RNN)

(0.17718148231506348, 0.5001199841499329)

---

## Q6 — Model B: LSTM

Build an **LSTM-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add an **LSTM layer** to capture long-term dependencies in sequences.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [19]:
# ============================================================
# Question Q6 — Model B: LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build an LSTM-based sequence model
# 2) Add Input, Embedding, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential LSTM model
lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm")

# TODO 6: Compile the model
lstm_model = compile_model(lstm_model, lr=1e-3)

# Print model summary
lstm_model.summary()

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,411,713 (5.39 MB)

 Trainable params: 1,411,713 (5.39 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# ============================================================
# Train and Evaluate LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the LSTM model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the LSTM model
history_lstm = train_model(lstm_model, run_name="proj5_lstm", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(lstm_model, "Validation (LSTM)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(lstm_model, "Test (LSTM)", test_ds)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 158s 496ms/step - accuracy: 0.5343 - loss: 0.1718 - val_accuracy: 0.7298 - val_loss: 0.1529 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 152s 487ms/step - accuracy: 0.6133 - loss: 0.1627 - val_accuracy: 0.6644 - val_loss: 0.1570 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 153s 489ms/step - accuracy: 0.7394 - loss: 0.1380 - val_accuracy: 0.7808 - val_loss: 0.1326 - learning_rate: 5.0000e-04
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 150s 478ms/step - accuracy: 0.7551 - loss: 0.1299 - val_accuracy: 0.6970 - val_loss: 0.1470 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 151s 484ms/step - accuracy: 0.8096 - loss: 0.1107 - val_accuracy: 0.7952 - val_loss: 0.1251 - learning_rate: 2.5000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 150s 481ms/step - accuracy: 0.8718 - loss: 0.0848 - val_accuracy: 0.8234 - val_loss: 0.1146 - learning_rate: 2.5000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 148s 474ms/step

(0.11547186225652695, 0.8256000280380249)

---

## Q7 — Model C: GRU

Build a **GRU-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add **Embedding → GRU → Dense(sigmoid)** layers.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [22]:
# ============================================================
# Question Q7 — Model C: GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a GRU-based sequence model
# 2) Add Input, Embedding, GRU, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential GRU model
gru_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: GRU layer
    layers.GRU(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="gru")

# TODO 6: Compile the model
gru_model = compile_model(gru_model, lr=1e-3)

# Print model summary
gru_model.summary()

Model: "gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,379,201 (5.26 MB)

 Trainable params: 1,379,201 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
# ============================================================
# Train and Evaluate GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the GRU model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the GRU model
history_gru = train_model(gru_model, run_name="proj5_gru", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(gru_model, "Validation (GRU)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(gru_model, "Test (GRU)", test_ds)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 155s 488ms/step - accuracy: 0.5174 - loss: 0.1730 - val_accuracy: 0.5336 - val_loss: 0.1717 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 150s 480ms/step - accuracy: 0.6022 - loss: 0.1630 - val_accuracy: 0.7882 - val_loss: 0.1256 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 150s 479ms/step - accuracy: 0.8556 - loss: 0.0894 - val_accuracy: 0.8708 - val_loss: 0.0887 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 147s 470ms/step - accuracy: 0.9394 - loss: 0.0452 - val_accuracy: 0.8588 - val_loss: 0.0987 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 204s 476ms/step - accuracy: 0.9779 - loss: 0.0187 - val_accuracy: 0.8662 - val_loss: 0.1481 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 201s 473ms/step - accuracy: 0.9939 - loss: 0.0074 - val_accuracy: 0.8560 - val_loss: 0.1749 - learning_rate: 2.5000e-04
Validation (GRU): loss=0.0887, acc=0.8708
Test (GRU): loss=0.0

(0.0913529321551323, 0.8527200222015381)

---

## 8) Complex Model D — Stacked LSTM with Dropout

In this question, build a **deeper LSTM-based sequence classifier** using two recurrent layers.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `Dropout(...)`
  - `LSTM(...)`
  - `Dense(1, activation="sigmoid")`
- Train the model using the same optimizer and callbacks.
- Evaluate on validation and test sets.
- Compare it with the single-layer LSTM from Q6.

### Goal
Study whether **stacking recurrent layers** helps the model learn richer sequential sentiment patterns.


---

In [25]:
# ============================================================
# Question Q8 — Complex Model D: Stacked LSTM + Dropout (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a stacked LSTM model with dropout
# 2) Add Input, Embedding, LSTM, Dropout, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential stacked LSTM model
stacked_lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: First LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: Dropout layer
    layers.Dropout(0.3),

    # TODO 6: Second LSTM layer
    layers.LSTM(RNN_UNITS // 2),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="stacked_lstm_dropout")

# TODO 8: Compile the model
stacked_lstm_model = compile_model(stacked_lstm_model, lr=1e-3)

# Print model summary
stacked_lstm_model.summary()

# TODO 9: Train the model
history_stacked_lstm = train_model(
    stacked_lstm_model,
    run_name="proj5_stacked_lstm_dropout",
    epochs=8
)

# TODO 10: Evaluate on validation set
evaluate_model(stacked_lstm_model, "Validation (Stacked LSTM + Dropout)", val_ds)

# TODO 11: Evaluate on test set
evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)

Model: "stacked_lstm_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 200, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 200, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,057 (5.57 MB)

 Trainable params: 1,461,057 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 283s 889ms/step - accuracy: 0.5344 - loss: 0.1730 - val_accuracy: 0.4938 - val_loss: 0.1766 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 317s 1s/step - accuracy: 0.5997 - loss: 0.1635 - val_accuracy: 0.6644 - val_loss: 0.1521 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 298s 935ms/step - accuracy: 0.7448 - loss: 0.1365 - val_accuracy: 0.7852 - val_loss: 0.1276 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 240s 765ms/step - accuracy: 0.8138 - loss: 0.1136 - val_accuracy: 0.7296 - val_loss: 0.1499 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 237s 755ms/step - accuracy: 0.8400 - loss: 0.1002 - val_accuracy: 0.7782 - val_loss: 0.1400 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 232s 740ms/step - accuracy: 0.8494 - loss: 0.0957 - val_accuracy: 0.7736 - val_loss: 0.1394 - learning_rate: 2.5000e-04
Validation (Stacked LSTM + Dropout): loss=0.1276, acc=0.7852
Test

(0.13491684198379517, 0.7633600234985352)

---

## 9) Complex Model E — Encoder–Decoder LSTM Classifier

In this question, build an **encoder–decoder style recurrent model** for sentiment classification.

### Idea
- The **encoder LSTM** reads the review and produces a compact context representation.
- A `RepeatVector` creates a short decoded sequence from that context.
- A **decoder LSTM** transforms the context into a richer hidden representation.
- A final dense layer predicts the review sentiment.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM` encoder
  - `RepeatVector(1)`
  - `LSTM` decoder
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it with the simpler one-layer LSTM.

### Goal
Explore whether a **more structured encoder–decoder design** is useful for sequence classification, even though it is more common in seq2seq tasks.


---

In [26]:
# ============================================================
# Question Q9 — Complex Model E: Encoder-Decoder LSTM Classifier (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define the input layer
# 2) Add Embedding, Encoder LSTM, RepeatVector, Decoder LSTM, and Dense layers
# 3) Build the functional model
# 4) Compile the model using the training utility
# 5) Train the model
# 6) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Define input layer
encdec_inputs = layers.Input(shape=(MAX_LEN,))

# TODO 2: Embedding layer
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(encdec_inputs)

# TODO 3: Encoder LSTM
encoded = layers.LSTM(RNN_UNITS)(x)

# TODO 4: Repeat encoded representation
decoded = layers.RepeatVector(1)(encoded)

# TODO 5: Decoder LSTM
decoded = layers.LSTM(RNN_UNITS // 2)(decoded)

# TODO 6: Output layer
encdec_outputs = layers.Dense(1, activation="sigmoid")(decoded)

# TODO 7: Build functional model
encdec_model = tf.keras.Model(encdec_inputs, encdec_outputs, name="encdec_lstm_classifier")

# TODO 8: Compile the model
encdec_model = compile_model(encdec_model, lr=1e-3)

# TODO 9: Print model summary
encdec_model.summary()

# TODO 10: Train the model
history_encdec = train_model(
    encdec_model,
    run_name="proj5_encdec_lstm",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(encdec_model, "Validation (Encoder-Decoder LSTM)", val_ds)

# TODO 12: Evaluate on test set
evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)

Model: "encdec_lstm_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_9 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,057 (5.57 MB)

 Trainable params: 1,461,057 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 157s 488ms/step - accuracy: 0.5248 - loss: 0.1727 - val_accuracy: 0.5836 - val_loss: 0.1680 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 152s 484ms/step - accuracy: 0.6201 - loss: 0.1621 - val_accuracy: 0.6784 - val_loss: 0.1605 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 203s 486ms/step - accuracy: 0.6688 - loss: 0.1537 - val_accuracy: 0.5648 - val_loss: 0.1677 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 153s 488ms/step - accuracy: 0.8052 - loss: 0.1108 - val_accuracy: 0.8120 - val_loss: 0.1143 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 154s 491ms/step - accuracy: 0.8647 - loss: 0.0847 - val_accuracy: 0.8202 - val_loss: 0.1081 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 153s 489ms/step - accuracy: 0.8979 - loss: 0.0687 - val_accuracy: 0.8562 - val_loss: 0.0944 - learning_rate: 5.0000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 152s 484ms/step - a

(0.11493930965662003, 0.8447200059890747)

---

## 10) Complex Model F — LSTM + GRU Hybrid

In this question, build a **hybrid recurrent architecture** that combines LSTM and GRU layers without using bidirectional processing.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `GRU(...)`
  - `Dropout`
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it against all earlier models in terms of accuracy and complexity.

### Goal
Test whether combining **LSTM-based memory** with a **GRU-based final sequence encoder** captures richer sentiment patterns than a single recurrent layer.


---

In [27]:
# ============================================================
# Question Q10 — Complex Model F: LSTM + GRU Hybrid (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a hybrid sequence model using LSTM and GRU layers
# 2) Add Input, Embedding, LSTM, GRU, Dropout, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential hybrid model
hybrid_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: GRU layer
    layers.GRU(RNN_UNITS // 2, dropout=0.2, recurrent_dropout=0.2),

    # TODO 6: Dropout layer
    layers.Dropout(0.3),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm_gru_hybrid")

# TODO 8: Compile the model
hybrid_model = compile_model(hybrid_model, lr=1e-3)

# TODO 9: Print model summary
hybrid_model.summary()

# TODO 10: Train the model
history_hybrid = train_model(
    hybrid_model,
    run_name="proj5_lstm_gru_hybrid",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(hybrid_model, "Validation (LSTM + GRU Hybrid)", val_ds)

# TODO 12: Evaluate on test set
evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)

Model: "lstm_gru_hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 200, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,448,897 (5.53 MB)

 Trainable params: 1,448,897 (5.53 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 254s 792ms/step - accuracy: 0.5229 - loss: 0.1733 - val_accuracy: 0.5606 - val_loss: 0.1704 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 254s 811ms/step - accuracy: 0.6391 - loss: 0.1612 - val_accuracy: 0.5266 - val_loss: 0.1741 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 254s 785ms/step - accuracy: 0.5566 - loss: 0.1714 - val_accuracy: 0.5754 - val_loss: 0.1689 - learning_rate: 5.0000e-04
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 243s 778ms/step - accuracy: 0.6457 - loss: 0.1538 - val_accuracy: 0.6562 - val_loss: 0.1589 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 244s 780ms/step - accuracy: 0.7014 - loss: 0.1464 - val_accuracy: 0.8160 - val_loss: 0.1321 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 261s 777ms/step - accuracy: 0.8396 - loss: 0.1029 - val_accuracy: 0.8248 - val_loss: 0.1085 - learning_rate: 5.0000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 242s 774ms/step

(0.10768669098615646, 0.8281599879264832)

---

## Q11 — Performance Comparison Table

Create a compact table comparing all completed models:

- Vanilla RNN
- LSTM
- GRU
- Stacked LSTM + Dropout
- Encoder–Decoder LSTM
- LSTM + GRU Hybrid

### Student Tasks
- Create a comparison table summarizing the results of all models.
- Include validation accuracy and test accuracy.
- Identify the best-performing recurrent model.
- Briefly comment on whether more complex architectures were helpful.


---

In [29]:
# ============================================================
# Question Q11 — Performance Comparison Table (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define helper functions for validation and training accuracy
# 2) Evaluate all trained models on the test dataset
# 3) Store model names and metrics in summary rows
# 4) Create a pandas DataFrame for comparison
# 5) Sort the table by test accuracy
# ============================================================

import pandas as pd

# TODO 1: Best validation accuracy from training history
def best_val_acc(history):
    return max(history.history.get("val_accuracy", [np.nan]))

# TODO 2: Final training accuracy from training history
def final_train_acc(history):
    return history.history.get("accuracy", [np.nan])[-1]

summary_rows = []

# TODO 3: Evaluate all models on the test set
_, rnn_test_acc = evaluate_model(rnn_model, "Test (Vanilla RNN)", test_ds)
_, lstm_test_acc = evaluate_model(lstm_model, "Test (LSTM)", test_ds)
_, gru_test_acc = evaluate_model(gru_model, "Test (GRU)", test_ds)
_, stacked_lstm_test_acc = evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)
_, encdec_test_acc = evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)
_, hybrid_test_acc = evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)

# TODO 4: Append summary rows
summary_rows.append(["Vanilla RNN", final_train_acc(history_rnn), best_val_acc(history_rnn), rnn_test_acc])
summary_rows.append(["LSTM", final_train_acc(history_lstm), best_val_acc(history_lstm), lstm_test_acc])
summary_rows.append(["GRU", final_train_acc(history_gru), best_val_acc(history_gru), gru_test_acc])
summary_rows.append(["Stacked LSTM + Dropout", final_train_acc(history_stacked_lstm), best_val_acc(history_stacked_lstm), stacked_lstm_test_acc])
summary_rows.append(["Encoder-Decoder LSTM", final_train_acc(history_encdec), best_val_acc(history_encdec), encdec_test_acc])
summary_rows.append(["LSTM + GRU Hybrid", final_train_acc(history_hybrid), best_val_acc(history_hybrid), hybrid_test_acc])

# TODO 5: Create DataFrame
results_df = pd.DataFrame(
    summary_rows,
    columns=["Model", "Final Train Acc", "Best Val Acc", "Test Acc"]
)

# TODO 6: Sort by test accuracy
results_df = results_df.sort_values(by="Test Acc", ascending=False).reset_index(drop=True)

# Display results
results_df

Test (Vanilla RNN): loss=0.1772, acc=0.5001
Test (LSTM): loss=0.1155, acc=0.8256
Test (GRU): loss=0.0914, acc=0.8527
Test (Stacked LSTM + Dropout): loss=0.1349, acc=0.7634
Test (Encoder-Decoder LSTM): loss=0.1149, acc=0.8447
Test (LSTM + GRU Hybrid): loss=0.1077, acc=0.8282


,Model,Final Train Acc,Best Val Acc,Test Acc
0,GRU,0.99390,0.8708,0.85272
1,Encoder-Decoder LSTM,0.95090,0.8628,0.84472
2,LSTM + GRU Hybrid,0.81315,0.8480,0.82816
3,LSTM,0.91260,0.8420,0.82560
4,Stacked LSTM + Dropout,0.84945,0.7852,0.76336
5,Vanilla RNN,0.63325,0.5060,0.50012


---

# Results and Discussions

Use the results above to answer the discussion questions below. You may revise the answers based on your actual experimental results (**Q1-Q11**).

---

## **Q12** — Which model performed best overall, and why might it outperform the others?  

**Answer:** ...
The GRU model performed the best with an 85.27% test accuracy. This is possible because GRU has gating mechanisms with fewer parameters. GRU models can remember long term context, is easier to optimize, is less prone to overfitting, and trains faster with more efficiency.

---

## **Q13** — Why does a vanilla RNN usually struggle more on long reviews?  

**Answer:** ...
Vanilla RNN struggles more on long reviews because it is prone to vanishing gradients. Vanishing gradients make it difficult to retain information from earlier parts of the sequence. As the review length increases, improtant data from the beginning can be lost before the model reaches the end of the execution and training.
---

## **Q14** — Why might a more complex model not always outperform a simpler GRU or LSTM?  

**Answer:** ...
More complex models may not always outperform a simpler GRU or LSTM model because additional layers and parameters make the model more difficult to train. They are also more prone to underfitting or overfitting. Complex models require more training data, carefeul hyperparameter tuning, and longer training times in order to reach accurate values. For this experiment, GRU was able to capture the important sequence while remaining "easy" to optimize.
---

## **Q15** — What is the main idea behind the encoder–decoder classifier used here?  

**Answer:** ...
The main idead behind the encoder-decoder classier used in this experiment is to encode the entire data input sequence into a compact latent representation using an ecoder LSTM and then use a decoder LSTM to process and exctract the representation before making a prediction. The decoder transforms the summary of the ecoder into features that are usefull for classification. This ecoder-decoder was the second best performing model at 84.47% test accuracy, displaying that the encoded representation retained useful information from the reviews.
---

### 🎉 Congratulations!

You have successfully completed **P5 — Sequence Modeling with RNNs**. Excellent work applying and comparing **RNN, LSTM, and GRU architectures** for a **sequence-based text classification** task using the **IMDB Movie Reviews dataset**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q11)** have been executed successfully
- All **Markdown responses (Q12–Q15)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.